# app18/edge — treinamento da Random Forest que vai rodar **dentro** do ESP32Mesmo dataset do `app17-7`, mesma leitura do InfluxDB, mesmo corte por rodada. O que mudaé o modelo e o destino.| Arquivo gerado | Vai para | Serve para ||---|---|---|| `ModeloMotorRF.hpp` | `edge/device/src/` | a floresta, em `if`/`else` || `ModeloMotorScaler.hpp` | `edge/device/src/` | a média e o desvio de cada feature || `modelo_motor_rf.pkl` | seu computador (ou `api/`) | a **mesma** floresta, em Python |Os dois `.hpp` saem do mesmo treino e viajam sempre juntos. O `.pkl` é a terceira cópia domesmo modelo — serve para guardar, reabrir depois e, se você quiser, rodar esta floresta naAPI do app18 no lugar da rede neural.

In [ ]:
# Versoes fixas: as mesmas do app17-7 e da api/ do app18, para o .pkl gerado aqui# abrir do outro lado. numpy e scikit-learn sao as que quebram o joblib.load se# divergirem. Os dois .hpp nao dependem de versao nenhuma: sao texto, C++ puro.## influxdb3-python le por SQL (Arrow Flight); pyarrow vem junto e e quem# converte o resultado em DataFrame. O micromlgen traduz a floresta em if/else.!pip install -q influxdb3-python pyarrow matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "joblib==1.5.3" "micromlgen==1.1.28"

## 1) Conectar no InfluxDB CloudOs mesmos valores do notebook do `app17-7` — é o mesmo dataset, lido do mesmo lugar.O `bucket` do Node-RED é o `database` aqui.

In [ ]:
import json, subprocess, zipfilefrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport joblibfrom influxdb_client_3 import InfluxDBClient3from sklearn.ensemble import RandomForestClassifierfrom sklearn.pipeline import Pipeline, make_pipelinefrom sklearn.preprocessing import StandardScalerfrom micromlgen import portINFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"INFLUX_BUCKET = "IoTSensores"          # o "database" no InfluxDB 3MEASUREMENT   = "vibracao_multiclasse"# A ORDEM desta lista e o contrato com o firmware: x[0] e mean_ax porque# mean_ax e a primeira coluna aqui. Mudar a ordem aqui sem mudar la faz o# modelo receber cada numero no lugar do outro.FEATURES = ["mean_ax", "mean_ay", "mean_az",            "std_ax", "std_ay", "std_az",            "std_mag", "p2p_mag"]CLASSES = ["operando", "inclinado_frente", "inclinado_tras", "anomalia"]client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)display(client.query("SHOW TABLES", language="sql").to_pandas())

## 2) Ler as janelasO mesmo `SELECT` do `app17-7`: as oito features, o rótulo e a rodada.

In [ ]:
colunas = ", ".join(f'"{f}"' for f in FEATURES)sql = f'''SELECT time, "label", "rodada", {colunas}FROM "{MEASUREMENT}"WHERE time >= now() - INTERVAL '30 days'ORDER BY time'''df = client.query(query=sql, language="sql").to_pandas()client.close()df = (df.query("label in @CLASSES")        .dropna(subset=FEATURES)        .sort_values("time")        .reset_index(drop=True))df["rodada"] = df["rodada"].astype(int)   # veio como tag, entao veio textoprint(f"{len(df)} janelas")print(df.groupby(["label", "rodada"]).size().to_string())

## 3) Split por rodada — a última rodada fica de fora do treinoIdêntico ao do notebook multiclasse. É de propósito: com o mesmo corte, a acurácia dafloresta pode ser comparada diretamente com a da rede neural que você já treinou.

In [ ]:
rodada_teste = df["rodada"].max()treino = df[df["rodada"] != rodada_teste]teste  = df[df["rodada"] == rodada_teste]print(f"treino: {len(treino)} janelas | teste: rodada {rodada_teste}, {len(teste)} janelas")

## 4) Treinar a florestaQuatro escolhas, e cada uma aparece depois em algum arquivo gerado:- **`y` em texto**, como no `app17-7`. Assim o `predict()` devolve o nome da classe e o  `.pkl` funciona na API do app18 sem nenhuma adaptação. No ESP32 é diferente: lá o  `predict()` do micromlgen devolve o **índice** da classe, e quem dá nome a ele é o vetor  `NOMES_CLASSES` do firmware, montado na seção 10 a partir de `classes_`.- **`StandardScaler` no Pipeline.** Ele faz **padronização**: `(valor − média) / desvio`,  deixando cada feature com média 0 e desvio 1. A árvore *não precisa* disso — ela compara  uma feature por vez com um limiar, e mudar a escala não muda a ordem dos valores. Está  aqui por **simetria** — é o mesmo `Scaler::standardize()` do outro app embarcado da  disciplina, e o aluno aplica a mesma sequência nos dois. O que **não** pode é treinar com  scaler e não padronizar no ESP32, ou o contrário: os limiares do header ficariam numa  escala e os dados em outra, e a predição sai errada sem aviso nenhum.- **15 árvores, número ímpar.** O desempate do micromlgen é por `>` estrito: num empate  vence o menor índice. Com 4 classes o ímpar não elimina todo empate (5-5-5 existe), mas  torna o caso raro — e a conferência da seção 9 mostra se algum aconteceu.- **`max_features=None`.** Por padrão cada divisão sorteia só √8 ≈ 2 features, o que obriga  a árvore a crescer fundo para compensar as divisões ruins. Com 8 features, deixar a  árvore olhar todas custa quase nada e cada corte passa a ser o melhor corte: as árvores  ficam rasas, o header cabe na tela e você reconhece a física nos limiares.

In [ ]:
# Pipeline com os passos nomeados "scaler" e "clf", como no Colab do app30 --# e os dois sao recuperados depois por named_steps, igualzinho la.modelo = Pipeline([    ("scaler", StandardScaler()),    ("clf",    RandomForestClassifier(n_estimators=15, max_features=None, random_state=42)),])modelo.fit(treino[FEATURES], treino["label"])scaler   = modelo.named_steps["scaler"]floresta = modelo.named_steps["clf"]# classes_ vem SEMPRE em ordem alfabetica, e e essa ordem que o indice do# micromlgen segue no ESP32: 0 = anomalia, 1 = inclinado_frente, e assim por diante.print("classes_ (a ordem dos indices no ESP32):", list(floresta.classes_))print("profundidade por arvore:", [a.tree_.max_depth for a in floresta.estimators_])

## 5) Avaliar

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,                             ConfusionMatrixDisplay)ORDEM = list(floresta.classes_)y_pred = modelo.predict(teste[FEATURES])print("acuracia:", round(accuracy_score(teste["label"], y_pred), 3))print("f1_macro:", round(f1_score(teste["label"], y_pred, average="macro"), 3))print(classification_report(teste["label"], y_pred, labels=ORDEM, zero_division=0))ConfusionMatrixDisplay.from_predictions(teste["label"], y_pred, labels=ORDEM,                                        xticks_rotation=45, cmap="Blues")plt.tight_layout(); plt.show()

## 6) A floresta contra a rede neural, no mesmo corteDa nuvem para a borda mudam **duas** coisas ao mesmo tempo: o modelo (rede neural →floresta) e o lugar onde ele roda (API → ESP32). Se as duas mudarem juntas sem medição,nenhuma comparação depois diz nada — uma diferença de resultado poderia vir de qualquer umdos dois lados.Esta célula resolve o eixo **modelo** aqui dentro, com os dois em Python, mesmos dados emesmo corte. Do firmware para a frente, muda só o **lugar**.

In [ ]:
from sklearn.neural_network import MLPClassifiermlp = make_pipeline(StandardScaler(),                    MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42))mlp.fit(treino[FEATURES], treino["label"])y_mlp = mlp.predict(teste[FEATURES])comparacao = pd.DataFrame({    "modelo":   ["MLP (a da API)", "Random Forest (a da borda)"],    "acuracia": [accuracy_score(teste["label"], y_mlp), accuracy_score(teste["label"], y_pred)],    "f1_macro": [f1_score(teste["label"], y_mlp, average="macro"),                 f1_score(teste["label"], y_pred, average="macro")],}).round(3)display(comparacao)

## 7) Quais features a floresta usouA floresta já traz essa conta pronta: cada divisão registra quanta impureza removeu, e aimportância é a soma disso por feature. Se `mean_ax` aparecer no topo, é a inclinação sendodetectada; se for `std_mag` ou `p2p_mag`, é a vibração.

In [ ]:
imp = pd.Series(floresta.feature_importances_, index=FEATURES).sort_values()imp.plot.barh(color="tab:green")plt.title("Importancia das features (Random Forest)")plt.tight_layout(); plt.show()print(imp.sort_values(ascending=False).round(3).to_string())

## 8) ExportarTrês arquivos, o mesmo modelo. O `.pkl` guarda o Pipeline inteiro (scaler + floresta) e éo que roda em Python; para o ESP32 são dois `.hpp`, porque o micromlgen porta **só afloresta** — o scaler fica de fora e precisa virar um header à parte.No ESP32 a ordem é sempre: ler as oito features, `Scaler::standardize()`, e só então`predict()`.

In [ ]:
pkl_filename        = "modelo_motor_rf.pkl"micromlgen_filename = "ModeloMotorRF.hpp"scaler_filename     = "ModeloMotorScaler.hpp"# --- o Pipeline inteiro, para Python ---joblib.dump(modelo, pkl_filename)# Confere recarregando, que e exatamente o que a API faz na inicializacao.recarregado = joblib.load(pkl_filename)assert np.array_equal(recarregado.predict(teste[FEATURES]), y_pred), \    "O .pkl recarregado nao reproduz o modelo avaliado."print(f"Arquivo {pkl_filename} gerado. Classes: {', '.join(recarregado.classes_)}")# --- a floresta, para o ESP32 ---# Folhas puras: cada folha vota numa classe so. E o que o micromlgen assume ao gerar# votes[argmax]. Se houver folha mista, o C++ pode decidir diferente do scikit-learn.for arvore in floresta.estimators_:    folhas = arvore.tree_.children_left == -1    contagens = arvore.tree_.value[folhas, 0, :]    assert (np.count_nonzero(contagens, axis=1) == 1).all(), \        "Folhas mistas: ha janelas praticamente identicas com rotulos diferentes. Confira a coleta."c_code = port(floresta)with open(micromlgen_filename, "w") as f:    f.write(c_code)# Sao ~350 linhas; as primeiras 25 ja mostram uma arvore inteira.print("\n".join(c_code.splitlines()[:25]))print(f"...  ({len(c_code.splitlines())} linhas no total)")print(f"\nArquivo {micromlgen_filename} gerado.")# --- o scaler, para o ESP32 ---means  = scaler.mean_.astype(float)scales = scaler.scale_.astype(float)# O template vai em texto normal e os numeros entram por replace. Com f-string# seria preciso dobrar TODA chave do C++ ({{ e }}) e a celula fica ilegivel:# nao da mais para ver onde termina o C++ e comeca o escape do Python.conteudo = '''#ifndef STANDARD_SCALER_HPP#define STANDARD_SCALER_HPP// StandardScaler ajustado no treino da Random Forest do app18/edge.// PADRONIZACAO: (input - means) / scales, deixando media 0 e desvio 1.// Ordem: mean_ax, mean_ay, mean_az, std_ax, std_ay, std_az, std_mag, p2p_mag.namespace Scaler {    const static float means[8] = {        __MEANS__    };    const static float scales[8] = {        __SCALES__    };    inline void standardize(const float* input, float* output) {        for (int i = 0; i < 8; i++) {            output[i] = (input[i] - means[i]) / scales[i];        }    }}#endif'''.replace("__MEANS__",  ", ".join(f"{v:.10f}f" for v in means)) \   .replace("__SCALES__", ", ".join(f"{v:.10f}f" for v in scales))with open(scaler_filename, "w") as f:    f.write(conteudo)print(conteudo)print(f"\nArquivo {scaler_filename} gerado.")

## 9) Conferir: o C++ decide igual ao Python?O header é compilado aqui mesmo e alimentado com as janelas de teste **já padronizadas**.Ele devolve o índice; `classes_` traduz o índice em nome, e aí dá para comparar com o`predict()` do scikit-learn. Se divergir, não trate os dois como o mesmo modelo — o quevocê embarcaria não seria o que você avaliou.

In [ ]:
codigo = r'''#include <iostream>#include <cstdint>#include "ModeloMotorRF.hpp"int main() {    Eloquent::ML::Port::RandomForest modelo;    float x[8];    while (std::cin >> x[0] >> x[1] >> x[2] >> x[3] >> x[4] >> x[5] >> x[6] >> x[7])        std::cout << modelo.predict(x) << std::endl;}'''Path("conferir.cpp").write_text(codigo, encoding="utf-8")subprocess.run(["g++", "-std=c++11", "conferir.cpp", "-o", "conferir"], check=True)X_teste_padr = scaler.transform(teste[FEATURES])entrada = pd.DataFrame(X_teste_padr).to_csv(index=False, header=False, sep=" ")saida = subprocess.run(["./conferir"], input=entrada, text=True, capture_output=True, check=True)indices_cpp = np.fromstring(saida.stdout, sep=" ", dtype=int)nomes_cpp   = floresta.classes_[indices_cpp]     # o indice vira nome pela MESMA ordemassert np.array_equal(nomes_cpp, y_pred), \    "C++ divergiu do Python: NAO embarque este header."print("Python e C++ concordam nas", len(nomes_cpp), "janelas de teste.")

## 10) A linha que vai para o firmwareO `predict()` do ESP32 devolve um número. Quem dá nome a ele é o vetor `NOMES_CLASSES` do`.cpp`, e a ordem tem que ser a de `classes_` — **alfabética**, que não é a ordem em quevocê pensa nas classes. Copie a linha impressa abaixo por cima da que está lá.Repare que `anomalia` cai no índice 0. Isso é bom: num empate de votos o micromlgen escolheo menor índice, então o erro raro acontece para o lado do alarme, e não para o lado de"está tudo normal".

In [ ]:
nomes = ", ".join(f'"{c}"' for c in floresta.classes_)print(f'const char* NOMES_CLASSES[4] = {{ {nomes} }};')print()for i, c in enumerate(floresta.classes_):    print(f"  indice {i} = {c}")

## 11) BaixarO ZIP traz os três arquivos.- os dois `.hpp` vão para `edge/device/src/`, por cima dos sintéticos que vieram no projeto- o `.pkl` é seu. Para rodar **esta** floresta na API do app18 em vez da rede neural,  coloque o arquivo na pasta `api/` e suba a API apontando para ele:  `MODELO_ARQUIVO=modelo_motor_rf.pkl uvicorn service_app:app --host 0.0.0.0 --port 8000`.  Aí o mesmo modelo está rodando nos dois lugares, e a comparação é só de **onde**.

In [ ]:
with zipfile.ZipFile("modelo_motor_rf.zip", "w") as pacote:    pacote.write(pkl_filename)    pacote.write(micromlgen_filename)    pacote.write(scaler_filename)try:    from google.colab import files    files.download("modelo_motor_rf.zip")except Exception:    print("modelo_motor_rf.zip gerado na pasta atual.")